In [1]:
!pip install segmentation-models patchify
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import segmentation_models as sm
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, concatenate


   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   --- ------------------------------------ 1.6/15.8 MB 12.0 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/15.8 MB 11.2 MB/s eta 0:00:02
   --------------- ------------------------ 6.0/15.8 MB 10.2 MB/s eta 0:00:01
   -------------------- ------------------- 8.1/15.8 MB 10.5 MB/s eta 0:00:01
   ------------------------- -------------- 10.2/15.8 MB 10.3 MB/s eta 0:00:01
   ------------------------------ --------- 12.1/15.8 MB 10.3 MB/s eta 0:00:01
   ------------------------------------- -- 14.7/15.8 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 10.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------------------------- ------- 2.4/2.9 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 11.2 MB/s eta 0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.2.6

 

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


ModuleNotFoundError: No module named 'tensorflow'

In [2]:
import os
import argparse
import json
import yaml
import sys
from datetime import datetime
from pathlib import Path
from multiprocessing import Pool
import urllib3

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import rasterio
import pystac
from tqdm import tqdm

from src.preprocess.s2_prep import cloud_mask_from_scl, apply_mask

urllib3.disable_warnings()

# AGGRESSIVE GDAL OPTIMIZATION SETTINGS
os.environ["GDAL_CACHEMAX"] = "2048"
os.environ["CPL_VSIL_CURL_CACHE_SIZE"] = "500000000"
os.environ["GDAL_HTTP_MULTIPLEX"] = "YES"
os.environ["GDAL_HTTP_VERSION"] = "2"
os.environ["CPL_VSIL_CURL_USE_CACHE"] = "YES"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.jp2"
os.environ["GDAL_HTTP_MAX_RETRY"] = "3"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"
os.environ["GDAL_NUM_THREADS"] = "ALL_CPUS"
os.environ["GDAL_TIFF_INTERNAL_MASK"] = "YES"
os.environ["GDAL_HTTP_CONNECTTIMEOUT"] = "10"
os.environ["GDAL_HTTP_TIMEOUT"] = "30"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

def load_cfg(p: str):
    p = Path(p)
    if not p.exists():
        sys.exit(f"Config file not found: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def item_time(it) -> datetime:
    dt_str = it["item"]["properties"]["datetime"].replace("Z", "")
    return datetime.fromisoformat(dt_str)

BAND_ALIASES = {
    "B02": ["blue", "B02", "blue-jp2"],
    "B03": ["green", "B03", "green-jp2"], 
    "B04": ["red", "B04", "red-jp2"],
    "B08": ["nir", "B08", "nir-jp2"],
    "SCL": ["scl", "SCL", "scl-jp2"],
}

def _choose_asset_href(stac_item: pystac.Item, key: str) -> str:
    key_up = key.upper()
    aliases = BAND_ALIASES.get(key_up, [key, key_up, key.lower()])
    
    for alias in aliases:
        if alias in stac_item.assets:
            href = stac_item.assets[alias].href.strip()
            if href.lower().endswith((".tif", ".tiff")):
                return href
    
    for alias in aliases:
        if alias in stac_item.assets:
            href = stac_item.assets[alias].href.strip()
            if href.lower().endswith(".jp2"):
                return href
    
    raise KeyError(f"Asset '{key}' not found. Available: {list(stac_item.assets.keys())}")

def open_da_pick_fast(item_dict: dict, asset_key_upper: str):
    """Ultra-fast data opening with minimal validation"""
    it = pystac.Item.from_dict(item_dict)
    href = _choose_asset_href(it, asset_key_upper)
    
    da = rxr.open_rasterio(
        href, 
        masked=True, 
        cache=False,
        chunks={'x': 2048, 'y': 2048},
        lock=False
    ).squeeze()
    
    if da.sizes.get('x', 0) == 0 or da.sizes.get('y', 0) == 0:
        raise ValueError(f"Empty raster for {asset_key_upper}")
    
    return da

def aoi_union(aoi_dir: Path) -> gpd.GeoDataFrame:
    aoi_dir = Path(aoi_dir)
    if not aoi_dir.exists():
        sys.exit(f"AOI directory not found: {aoi_dir}")
    files = list(aoi_dir.glob("*.geojson"))
    if not files:
        sys.exit(f"No AOI files in {aoi_dir}")
    gdfs = []
    for f in files:
        g = gpd.read_file(f)
        g = g[g.geometry.notna()]
        if not g.empty:
            gdfs.append(g.to_crs("EPSG:4326"))
    if not gdfs:
        sys.exit("AOI files contain no valid geometries.")
    merged = pd.concat(gdfs, ignore_index=True).explode(ignore_index=True)
    u = merged.geometry.union_all() if hasattr(merged.geometry, "union_all") else merged.geometry.unary_union
    return gpd.GeoDataFrame(geometry=[u], crs="EPSG:4326")

def pair_nearest(t0_items, t1_items):
    if not t0_items or not t1_items: 
        return []
    t1t = [(item_time(i1), i1) for i1 in t1_items]
    out = []
    for i0 in t0_items:
        t0t = item_time(i0)
        best = min(t1t, key=lambda x: abs(x[0]-t0t))[1]
        out.append((i0, best))
    return out

def process_pair(args):
    # Set aggressive GDAL settings in each worker
    os.environ["GDAL_CACHEMAX"] = "2048"
    os.environ["CPL_VSIL_CURL_CACHE_SIZE"] = "500000000"
    os.environ["GDAL_HTTP_MULTIPLEX"] = "YES"
    os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"
    
    i, i0, i1, aoi, ts, stride, res, chips_dir = args
    rows = []
    
    try:
        print(f"⚡ Processing pair {i}...")
        
        # Read t1 bands (reference)
        b02_1 = open_da_pick_fast(i1["item"], "B02")
        b03_1 = open_da_pick_fast(i1["item"], "B03") 
        b04_1 = open_da_pick_fast(i1["item"], "B04")
        b08_1 = open_da_pick_fast(i1["item"], "B08")
        scl1 = open_da_pick_fast(i1["item"], "SCL")
        
        scl1 = scl1.rio.reproject_match(b02_1, resampling=rasterio.enums.Resampling.nearest)
        
        # Read t0 bands - only reproject if different CRS
        b02_0 = open_da_pick_fast(i0["item"], "B02")
        b03_0 = open_da_pick_fast(i0["item"], "B03")
        b04_0 = open_da_pick_fast(i0["item"], "B04") 
        b08_0 = open_da_pick_fast(i0["item"], "B08")
        scl0 = open_da_pick_fast(i0["item"], "SCL")
        
        if b02_0.rio.crs != b02_1.rio.crs:
            b02_0 = b02_0.rio.reproject_match(b02_1)
            b03_0 = b03_0.rio.reproject_match(b03_1) 
            b04_0 = b04_0.rio.reproject_match(b04_1)
            b08_0 = b08_0.rio.reproject_match(b08_1)
            scl0 = scl0.rio.reproject_match(b02_1, resampling=rasterio.enums.Resampling.nearest)

        # Clip to AOI
        aoi_ref = aoi.to_crs(scl1.rio.crs)
        
        try:
            b02_1 = b02_1.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b03_1 = b03_1.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b04_1 = b04_1.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b08_1 = b08_1.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            scl1 = scl1.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            
            b02_0 = b02_0.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b03_0 = b03_0.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b04_0 = b04_0.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            b08_0 = b08_0.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
            scl0 = scl0.rio.clip(aoi_ref.geometry, aoi_ref.crs, drop=True, from_disk=True)
        except:
            print(f"  ⚠️ Clipping failed, using original bounds")

        # Stack and process
        t1_stack = np.stack([b02_1.values, b03_1.values, b04_1.values, b08_1.values])
        t0_stack = np.stack([b02_0.values, b03_0.values, b04_0.values, b08_0.values])

        # Apply cloud masks
        keep1 = cloud_mask_from_scl(scl1)
        keep1[np.isnan(scl1.values)] = True
        keep0 = cloud_mask_from_scl(scl0) 
        keep0[np.isnan(scl0.values)] = True
        
        t1_stack = apply_mask(t1_stack, keep1)
        t0_stack = apply_mask(t0_stack, keep0)

        H, W = t1_stack.shape[1:]
        if H < ts or W < ts:
            print(f"  ⚠️ Pair {i} too small ({H}x{W}), skipping")
            return []

        transform = b02_1.rio.transform()
        chip_id = 0
        
        for y in range(0, H - ts + 1, stride):
            for x in range(0, W - ts + 1, stride):
                c1 = t1_stack[:, y:y+ts, x:x+ts]
                c0 = t0_stack[:, y:y+ts, x:x+ts]
                
                # Quick validity check
                if np.isfinite(c1).mean() < 0.1 or np.isfinite(c0).mean() < 0.1:
                    continue
                    
                # Save chips
                out0 = chips_dir / f"s2_t0_{i}_{chip_id}.npy"
                out1 = chips_dir / f"s2_t1_{i}_{chip_id}.npy"
                
                np.save(out0, c0.astype("float32"))
                np.save(out1, c1.astype("float32"))
                
                # Calculate bounds
                x0_map, y0_map = transform * (x, y)
                x1_map, y1_map = transform * (x + ts, y + ts)
                xmin, xmax = sorted([x0_map, x1_map])
                ymin, ymax = sorted([y0_map, y1_map])
                
                rows.append(dict(
                    chip_id=f"s2_{i}_{chip_id}",
                    split="train",
                    t0_npy=str(out0.resolve()),
                    t1_npy=str(out1.resolve()),
                    xmin=float(xmin), ymin=float(ymin),
                    xmax=float(xmax), ymax=float(ymax),
                    width=ts, height=ts, res=res, crs=str(b02_1.rio.crs)
                ))
                chip_id += 1
        
        print(f"  ✅ {chip_id} chips for pair {i}")
        return rows
        
    except Exception as e:
        print(f"❌ Pair {i} failed: {str(e)[:100]}")
        return []

def main(config, items_json, t0_start, t0_end, t1_start, t1_end, out_index, tile, stride, num_workers):
    cfg = load_cfg(config)
    items_json = Path(items_json)
    items = json.loads(items_json.read_text())

    s2 = [it for it in items if it.get("collection") == "sentinel-2-l2a"]
    print(f"⚡ Found {len(s2)} S2 items")

    t0s, t0e = datetime.fromisoformat(t0_start), datetime.fromisoformat(t0_end)
    t1s, t1e = datetime.fromisoformat(t1_start), datetime.fromisoformat(t1_end)
    
    # AGGRESSIVE FILTERING - Low cloud cover only
    t0 = [it for it in s2 if t0s <= item_time(it) <= t0e and 
          it["item"]["properties"].get("eo:cloud_cover", 100) < 15]
    t1 = [it for it in s2 if t1s <= item_time(it) <= t1e and 
          it["item"]["properties"].get("eo:cloud_cover", 100) < 15]

    pairs = pair_nearest(t0, t1)
    print(f"⚡ {len(pairs)} pairs (cloud<15%)")

    if not pairs:
        sys.exit("No pairs found!")

    aoi = aoi_union(Path(cfg["paths"]["aoi_dir"]))
    chips_dir = Path(cfg["paths"]["chips_dir"]).resolve()
    chips_dir.mkdir(parents=True, exist_ok=True)

    res = float(cfg["preprocess"].get("resolution", 10))
    args_list = [(i, i0, i1, aoi, tile, stride, res, chips_dir) for i, (i0, i1) in enumerate(pairs)]

    # PARALLEL PROCESSING
    print(f"⚡ Processing with {num_workers} workers...")
    with Pool(num_workers) as p:
        results = list(tqdm(p.imap(process_pair, args_list), total=len(args_list)))

    rows = [r for sub in results for r in sub]
    df = pd.DataFrame(rows)

    if df.empty:
        print("❌ No chips created")
    else:
        df.to_parquet(out_index)
        print(f"✅ {len(df)} chips → {out_index}")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--items", required=True)
    ap.add_argument("--t0", nargs=2, required=True)
    ap.add_argument("--t1", nargs=2, required=True)
    ap.add_argument("--out_index", required=True)
    ap.add_argument("--tile_size", type=int, default=128)  # Smaller default
    ap.add_argument("--stride", type=int, default=256)     # Larger stride
    ap.add_argument("--num_workers", type=int, default=8)  # More workers

    a = ap.parse_args()
    main(a.config, a.items, a.t0[0], a.t0[1], a.t1[0], a.t1[1], 
         a.out_index, a.tile_size, a.stride, a.num_workers)

ModuleNotFoundError: No module named 'src'